# Likelihood weighting

We have previously introduce the *discrete sampling* method and shown how it can be adapted for conditional distributions using *rejection sampling*. In this technique, we sample randomly from across the parameter space and then only retain those samples which have the correct values of the *evidence variables*: variables that are considered to be fixed by our accumulated knowledge, or by our desired quantity. This will be clearer with an example.

Let's say that we want to compute the probability that someone with a career in programming but not in the finance sector studies mathematics, $P(M\vert P=True, F=False)$. In rejection sampling we would allow the sampler to generate events with $P$ both True and False, and then reject all those samples where $P$ is False. Can we compute only those values that we would need to do this? Yes, with an approach called *likelihood reweighting* in which we fix certain variables to the desired values and then reweight the answers in a compensate for the fact that the variable don't have those values all of the time.

An alternative approach is to only generate samples that are consistent with the evidence, that is, that have the same value as the evidence variables. This means that no events are wasted which is good. However, we cannot just fix those values because this would bias the outcome. To see this, think about how to do this for $P(M\vert P=True, F=False)$. Here, $P$ and $F$ are so-called evidence variables (image we had "observed" them and found that they took these values: they are evidence). We could just set $P=True$, $F=False$, but what if $P(F=False)=0$? In this extreme case the conditions we are simulating could never occur. We need to compensate for this. In outline, we do this as follows:

* Fix all the evidence variables to their observed values
* Sample non-evidence variables conditioned on the fixed evidence variables
* Compute the probability that the sample you have just drawn would give rise to the evidence that you have fixed.
* Include your sample in the trace, weighted by this probability.

Let's illustrate this by working through our example.

* Fix $P=True$.
* Set weight $w=1.0$.
* $G$ is an evidence variable and is fixed at False. Set $w\leftarrow w\times P(G=False) = 0.6$.
* $M$ is not evidence. Sample from $P(M) = (0.75,0.25)$, suppose this returns True.
* Sample from $P(F\vert M=True) = (0.4,0.6)$, suppose this returns False.
* $P$ is an evidence variable and is True. Set $w\leftarrow w\times P(P=True\vert M=True) = 0.6\times 0.55 = 0.33$.
* Record $(G,M,F,P) = (False, True, False, True)$ with weight 0.33.
* Count samples as before, scaled by the weights assigned to each row of the trace.

Why does this work? Firstly, all of the samples generate have the observed values of the evidence variables and so all of the samples can be used. This give improved accuracy results or faster convergens for a given level of accuracy. Secondly, it can be shown rigourously that weighting of the likelihoods in this way is fully consistent with direct sampling from the desired posterior distribution. A proof can be found in Section 14.5.1 of Russell and Norvig (Fifth Edition). The main disadvantage is the flexibility: we have to fix the evidences variables, and if we want to compute for a different set of evidence then we have to regenerate a new trace.

Let's implement this. First, let's do the calculation:



In [1]:

import numpy as np

PG = np.array([0.6,0.4])
PM = np.array([0.75,0.25])
PF_M = np.array([[0.8,0.4],[0.2,0.6]])
PP_M = np.array([[0.65,0.45],[0.35,0.55]]) 

Let us first generate something we know. We'll compute $P(\lnot F\vert M))$ We need to set $M=1$ as evidence.

In [2]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Set M=1 as evidence and reweight
    M = 1
    w *= PM[M]
    # Draw one sample from P(F|M)
    F = np.random.binomial(1,PF_M[1,M])
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence (ie conditioned on M=1) by construction, so we do the following
# Find all rows in the trace where F=0
FEq0MEq1 = Trace[np.where(Trace[:,2]==0)]
# Sum the weights in the subset of the trace we want, and divide by the sum of the weights in the full trace
PFEq0MEq1 = (FEq0MEq1[:,4].sum()) / Trace[:,4].sum()
print(f"P(F=0|M=1) = {PFEq0MEq1} (expected {PF_M[0,1]})")


P(F=0|M=1) = 0.4014 (expected 0.4)


Check another one of the inputs.

In [3]:
print(f"P(F=1|M=1) = {Trace[np.where(Trace[:,2]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PF_M[1,1]})")
print(f"P(P=0|M=1) = {Trace[np.where(Trace[:,3]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PP_M[0,1]})")
print(f"P(P=1|M=1) = {Trace[np.where(Trace[:,3]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PP_M[1,1]})")


P(F=1|M=1) = 0.5986 (expected 0.6)
P(P=0|M=1) = 0.4475 (expected 0.45)
P(P=1|M=1) = 0.5525 (expected 0.55)


Now something that isn't an input: $P(M\vert F)$. We can compute exactly to check:

In [5]:
PF = PF_M @ PM
PM_F = (PF_M*PM).T/PF
print(f"P(M|F) = {PM_F}")

P(M|F) = [[0.85714286 0.5       ]
 [0.14285714 0.5       ]]


First fix $F=0$

In [6]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Sample M
    M  = np.random.binomial(1,PM[1])
    # Set evidence
    F = 0
    w *= PF_M[F,M]
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence by construction, so all we need to do is count the rows where $M=0$
print(f"P(M=0|F=0) = {Trace[np.where(Trace[:,1]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,0]})")
print(f"P(M=1|F=0) = {Trace[np.where(Trace[:,1]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,0]})")

P(M=0|F=0) = 0.8556388396177835 (expected 0.8571428571428572)
P(M=1|F=0) = 0.1443611603822166 (expected 0.8571428571428572)


Now for F=1

In [7]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Sample M
    M  = np.random.binomial(1,PM[1])
    # Set evidence
    F = 1
    w *= PF_M[F,M]
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence by construction, so all we need to do is count the rows where $M=0$
print(f"P(M=0|F=1) = {Trace[np.where(Trace[:,1]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,1]})")
print(f"P(M=1|F=1) = {Trace[np.where(Trace[:,1]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,1]})")

P(M=0|F=1) = 0.5050924685071025 (expected 0.5)
P(M=1|F=1) = 0.49490753149289723 (expected 0.5)


Finally, let's compute $P(M\vert \lnot F,P)$. This is actually rather tricky to compute by hand, so let's compute by direct sampling:

In [8]:
N = 10000
Trace = np.zeros([N,4])
for i in range(N):
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Draw one sample from P(M)
    M = np.random.binomial(1,PM[1])
    # Draw one sample from P(F|M)
    F = np.random.binomial(1,PF_M[1,M])
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P])

# Rows where F=0
x = Trace[np.where(Trace[:,2]==0)]
# Rows where F=0 and P=1
x = x[np.where(x[:,3]==1)]

PM_NotFP = x[np.where(x[:,1]==1)].shape[0]/x.shape[0]
PNotM_NotFP = x[np.where(x[:,1]==0)].shape[0]/x.shape[0]

print(f"P(M=0|F=0,P=1) = {PM_NotFP}")
print(f"P(M=0|F=0,P=1) = {PNotM_NotFP}")

P(M=0|F=0,P=1) = 0.19732313575525812
P(M=0|F=0,P=1) = 0.8026768642447418


In [10]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Sample M
    M  = np.random.binomial(1,PM[1])
    # Set evidence
    F = 0
    w *= PF_M[F,M]
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    P = 1
    w *= PP_M[P,M]
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence by construction, so all we need to do is count the rows where $M=0$
print(f"P(M=0|F=0,P=1) = {Trace[np.where(Trace[:,1]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PNotM_NotFP})")

print(f"P(M=1|F=0,P=1) = {Trace[np.where(Trace[:,1]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PM_NotFP})")


P(M=0|F=0,P=1) = 0.7971838205364006 (expected 0.8026768642447418)
P(M=1|F=0,P=1) = 0.2028161794635992 (expected 0.19732313575525812)
